In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F

# 1. Generate one row per day for 2026
dates_df = spark.sql("""
    SELECT explode(sequence(to_date('2026-01-01'), to_date('2026-12-31'), interval 1 day)) AS FullDate
""")

# 2. Derive the dimension columns
dimdate_df = dates_df.select(
    F.date_format("FullDate", "yyyyMMdd").cast("int").alias("DateKey"),
    F.col("FullDate"),
    F.year("FullDate").alias("Year"),
    F.quarter("FullDate").alias("Quarter"),
    F.month("FullDate").alias("MonthNumber"),
    F.date_format("FullDate", "MMMM").alias("MonthName"),
    F.dayofmonth("FullDate").alias("DayOfMonth"),
    F.dayofweek("FullDate").alias("DayOfWeek"),
    F.date_format("FullDate", "EEEE").alias("DayName"),
    F.weekofyear("FullDate").alias("WeekNumber"),
    F.when(F.dayofweek("FullDate").isin(1, 7), 1).otherwise(0).alias("IsWeekend"),
)

# 3. Check the result before writing
dimdate_df.printSchema()
display(dimdate_df.limit(10))

# 4. Write back to the lakehouse as a managed Delta table
(dimdate_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dimdate"))

StatementMeta(, 8d195e6a-59b4-4402-ac72-ab25f4e9f9b7, 3, Finished, Available, Finished, False)

root
 |-- DateKey: integer (nullable = true)
 |-- FullDate: date (nullable = false)
 |-- Year: integer (nullable = false)
 |-- Quarter: integer (nullable = false)
 |-- MonthNumber: integer (nullable = false)
 |-- MonthName: string (nullable = false)
 |-- DayOfMonth: integer (nullable = false)
 |-- DayOfWeek: integer (nullable = false)
 |-- DayName: string (nullable = false)
 |-- WeekNumber: integer (nullable = false)
 |-- IsWeekend: integer (nullable = false)



SynapseWidget(Synapse.DataFrame, 00a2e675-d319-4fd2-b36a-47306e873b56)

In [2]:
display(dimdate_df.orderBy(F.col("FullDate").desc()).limit(5))

StatementMeta(, 8d195e6a-59b4-4402-ac72-ab25f4e9f9b7, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 93a492d6-4274-44b5-8e46-f1222d2a245a)